In [2]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
gdf_2022 = gpd.read_file('2022data.shp')
gdf_2022 = gdf_2022.to_crs("32718")

In [ ]:
df_water_2024 = pd.read_csv('water_samples_2024.csv')

There are X and Y coordinates for the location of points, I'm gonna use that to make a geodataframe. I don't have access to any kind of
.gpkg or .shp file for the 2024 data so relying on this CSV will have to do

In [15]:
gdf_2024 = gpd.GeoDataFrame(
    df_water_2024,
    geometry=gpd.points_from_xy(df_water_2024["x"], df_water_2024["y"]),
    crs='EPSG:32718',
)

In [ ]:
gdf_2026 = gpd.read_file('2026_water_samples_geo.gpkg')

c:\Users\saric\Documents\Code\watergroup\venv\lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non-conformant content for record 1 in column CreationDate, 2026-06-14T17:25:28.0Z, successfully parsed
  return ogr_read(


In [22]:
# simplifying some column names
gdf_2026 = gdf_2026.rename(columns={
    "Tree_cover": "tree_cover", 
    "Shrubs_cover": "shrub_cover", 
    "SucculentsCacti_cover": "succulent_cover",
    "Grasses_cover": "grass_cover",
    "Herbaceous_plant_cover": "herbaceous_cover",
    "Bare_cover": "bare_cover",
    "Electrical_Conductivity": "EC",
    'Al\n(mg/L)': 'Al',
    'As\n(mg/L)': 'As',
    'Ba\n(mg/L)': 'Ba',
    'Ca\n(mg/L)': 'Ca',
    'Cd\n(mg/L)': 'Cd',
    'Cu\n(mg/L)': 'Cu',
    'Fe\n(mg/L)': 'Fe',
    'K\n(mg/L)': 'K',
    'Mg\n(mg/L)': 'Mg',
    'Na\n(mg/L)': 'Na',
    'Pb\n(mg/L)': 'Pb',
    'S\n(mg/L)': 'S',
})

In [23]:
gdf_2026.columns

Index(['GlobalID', 'CreationDate', 'Creator', 'EditDate', 'Editor',
       'Additional_notes', 'tree_cover', 'shrub_cover', 'succulent_cover',
       'grass_cover', 'herbaceous_cover', 'bare_cover', 'Water_Temperature',
       'EC', 'pH', 'ID', 'Date_and_time', 'geometry'],
      dtype='object')

In [6]:
element_cols = ['Al', 'As', 'Ba', 'Ca', 'Cd', 'Cu', 'Fe', 'K', 'Mg', 'Na', 'Pb', 'S']
vegetation_cover_columns = ['tree_cover', 'shrub_cover', 'succulent_cover', 'grass_cover', 'herbaceous_cover', 'bare_cover']

In [7]:
# so we entered the data in a little bit of an insane way.
# For some samples, we did observe vegetation surrounding the sample. For these, if a category had 0% cover, we either entered a 0 value,
# or no value at all (a NaN).
# for the samples where we didn't collect anything, every category for vegetation cover is a NaN
# I can't just make all NaNs 0, because it would be wrong: we just don't have data for some of these, so they aren't 0. They would
# mess up any model we might make. but I also need to replace the NaNs with 0s on the data points we DID collect but didn't input a 0 in

In [8]:
# checks if any of the veg cover values isn't a NaN across each row
measured_vegetation = df_water_2026[vegetation_cover_columns].notna().any(axis=1)
df_water_2026['measured_vegetation'] = measured_vegetation
df_water_2026.loc[measured_vegetation, vegetation_cover_columns] = (
    df_water_2026.loc[measured_vegetation, vegetation_cover_columns].fillna(0)
)

In [9]:
df_measured_vegetation = df_water_2026[measured_vegetation].copy()

In [10]:
df_measured_vegetation[vegetation_cover_columns].sum(axis=1) - df_measured_vegetation['bare_cover']

0      58.0
2     109.0
5      43.0
6      64.0
16    115.0
19     86.0
22     93.0
61    141.0
63     95.0
72     44.0
73     80.0
76     94.0
dtype: float64